# convtranspose-bn-activation-block composite — cx1: DCGAN G block packaged as an nn.Module subclass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `convtranspose-bn-activation-block`, `nn-module-subclass`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "convtranspose-bn-activation-block"
DD_ATOM_IDS = ["convtranspose-bn-activation-block", "nn-module-subclass"]
DD_SUBTOPICS = ["GAN: ConvT+BN+Activation block", "PyTorch: nn.Module subclassing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The DCGAN generator repeats one structural unit over and over: a `ConvTranspose2d -> BatchNorm2d -> ReLU` triple that doubles spatial size and halves channels. Two atoms wire together here:

1. **convtranspose-bn-activation-block** — the layer triple itself. Canonical hyperparams are `kernel_size=4, stride=2, padding=1, bias=False`. Output size: `H_out = (H_in - 1) * stride - 2 * padding + kernel = 2 * H_in`. `bias=False` because the immediately-following `BatchNorm2d` re-centres anyway, and ReLU (not LeakyReLU) is the generator-side activation per the DCGAN paper.
2. **nn-module-subclass** — wrapping the triple as a `class GBlock(nn.Module): ...` instead of just returning `nn.Sequential(...)`. The subclass discipline is: call `super().__init__()` first; register sub-Modules by attribute assignment (`self.convt = ...`); implement `forward(self, x)`; never call `.forward()` directly (use `block(x)`).

**Anatomy.**
```python
class GBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()                                  # nn-module-subclass.
        self.convt = nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False)
        self.bn    = nn.BatchNorm2d(out_c)                  # convtranspose-bn-activation-block.
        self.act   = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.bn(self.convt(x)))
```

**Why subclass instead of `nn.Sequential`.** Subclassing lets the block hold extra state (e.g. a flag, a running counter, a custom init method), expose named children (`block.convt.weight`), and override `extra_repr` for nicer printing. ARENA's generator uses `nn.Sequential` of these blocks as the OUTER stack, but each block itself is typically a named subclass once you want to do init or surgery on it.

### Composite Exercise — DCGAN G block packaged as an nn.Module subclass

**Atoms exercised together**: `convtranspose-bn-activation-block`, `nn-module-subclass`

Implement `cx1_make_g_block_cls()` — return a `GBlock` class (an `nn.Module` subclass) with the following contract:

- `GBlock(in_channels: int, out_channels: int)` — constructor.
- Inside `__init__`, call `super().__init__()` FIRST (atom: nn-module-subclass).
- Register three children as attributes (NOT in a `nn.Sequential`):
  - `self.convt = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)`
  - `self.bn = nn.BatchNorm2d(out_channels)`
  - `self.act = nn.ReLU(inplace=True)`
- `forward(self, x)` runs `convt -> bn -> act` in order (atom: convtranspose-bn-activation-block).

The test checks:
1. Returned value is a CLASS, not an instance.
2. Constructed instance is an `nn.Module` (subclass discipline).
3. Children are named `convt`, `bn`, `act` and are of the right types.
4. ConvT bias is `None` (bias=False was respected).
5. Input `(N, in_c, H, W)` produces output `(N, out_c, 2*H, 2*W)`.
6. `forward` output is non-negative (ReLU).
7. `block(x)` (i.e. `__call__`) and `block.forward(x)` agree numerically.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx1_make_g_block_cls():
    """Return the GBlock class (an nn.Module subclass)."""
    raise NotImplementedError

def _test_cx1():
    GBlock = cx1_make_g_block_cls()

    # Case A: returned a class, not an instance.
    assert isinstance(GBlock, type), f'must return a class; got {type(GBlock).__name__}'

    # Case B: an instance is an nn.Module.
    block = GBlock(in_channels=8, out_channels=4)
    assert isinstance(block, nn.Module), 'GBlock must subclass nn.Module'

    # Case C: child modules registered by canonical names + types.
    children = dict(block.named_children())
    assert set(children.keys()) == {'convt', 'bn', 'act'}, (
        f"expected named children {{convt,bn,act}}; got {sorted(children.keys())}"
    )
    assert isinstance(children['convt'], nn.ConvTranspose2d), (
        f'convt must be nn.ConvTranspose2d; got {type(children["convt"]).__name__}'
    )
    assert isinstance(children['bn'], nn.BatchNorm2d)
    assert isinstance(children['act'], nn.ReLU)

    # Case D: ConvT hyperparams + bias=False.
    ct = children['convt']
    assert ct.in_channels == 8 and ct.out_channels == 4
    assert ct.kernel_size == (4, 4) and ct.stride == (2, 2) and ct.padding == (1, 1)
    assert ct.bias is None, 'ConvT bias must be None (bias=False)'

    # Case E: spatial doubling + channel halving on a forward pass.
    x = t.randn(2, 8, 4, 4)
    y = block(x)
    assert y.shape == (2, 4, 8, 8), f'expected (2,4,8,8); got {tuple(y.shape)}'

    # Case F: ReLU non-negativity.
    assert (y >= 0).all(), 'output must be non-negative (ReLU)'

    # Case G: __call__ and .forward agree (subclass uses Module machinery).
    block.eval()
    x2 = t.randn(1, 8, 4, 4)
    with t.no_grad():
        a = block(x2)
        b = block.forward(x2)
    assert t.allclose(a, b, atol=1e-7), '__call__ and .forward disagree — likely overrode __call__ instead of forward'

    # Case H: cross-check vs an equivalent nn.Sequential composition (in eval mode so BN matches).
    ref = nn.Sequential(
        nn.ConvTranspose2d(8, 4, 4, 2, 1, bias=False),
        nn.BatchNorm2d(4),
        nn.ReLU(inplace=True),
    )
    ref[0].load_state_dict(children['convt'].state_dict())
    ref[1].load_state_dict(children['bn'].state_dict())
    ref.eval()
    with t.no_grad():
        expected = ref(x2)
    assert t.allclose(a, expected, atol=1e-6), 'block output disagrees with the canonical ConvT->BN->ReLU sequence'
    _dd_passed.add('cx1')

_test_cx1()

<details><summary>Show solution — cx1</summary>

```python
def cx1_make_g_block_cls():
    class GBlock(nn.Module):
        def __init__(self, in_channels, out_channels):
            # Atom B (nn-module-subclass): super().__init__() FIRST.
            super().__init__()
            # Atom A (convtranspose-bn-activation-block): ConvT(bias=False) -> BN -> ReLU.
            self.convt = nn.ConvTranspose2d(
                in_channels, out_channels,
                kernel_size=4, stride=2, padding=1, bias=False,
            )
            self.bn = nn.BatchNorm2d(out_channels)
            self.act = nn.ReLU(inplace=True)

        def forward(self, x):
            return self.act(self.bn(self.convt(x)))

    return GBlock
```

`super().__init__()` wires up `_parameters`, `_modules`, `_buffers` — without it, attribute assignment still works but the Module machinery (`.parameters()`, `.state_dict()`, `.to(device)`) is broken. The `bias=False` choice isn't free — the ConvT bias would just be subtracted away by BN's running mean, so it's wasted parameters. Keep `inplace=True` on ReLU only when you don't need the pre-activation value for autograd elsewhere (here it's safe because we don't branch off the conv output).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["GAN: ConvT+BN+Activation block", "PyTorch: nn.Module subclassing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()